# 01 - On-Time Performance Heatmap

This notebook generates a GitHub-style calendar heatmap showing daily on-time performance.

**Features:**
- Calendar heatmap with color-coded days
- Hover tooltips with detailed stats
- Multi-year support
- Dark theme styling

**Output:** `static/plots/on_time_heatmap_{year}.html`

In [10]:
import os
import sys
import sqlite3
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime, timedelta

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..','..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Database path
DB_PATH = os.path.join(project_root, 'data', 'caltrain_lat_long.db')
GTFS_PATH = os.path.join(project_root, 'gtfs_data')
OUTPUT_DIR = os.path.join(project_root, 'static', 'plots')

print(f"Database: {DB_PATH}")
print(f"Exists: {os.path.exists(DB_PATH)}")

Database: d:\caltrain-tracker\data\caltrain_lat_long.db
Exists: True


## Configuration

Adjust these settings to customize the heatmap:

In [11]:
# ====================
# CONFIGURATION
# ====================

YEAR = 2025  # Year to display

# Color scheme (GitHub-inspired dark theme)
COLORSCALE = [
    [0.0, '#161b22'],   # No data / very poor
    [0.01, '#161b22'],
    [0.02, '#da3633'],  # Poor (< 50%)
    [0.25, '#f85149'],  # Below average
    [0.50, '#d29922'],  # Average (~70%)
    [0.75, '#3fb950'],  # Good (~85%)
    [1.0, '#238636']    # Excellent (95%+)
]

# Theme colors
THEME = {
    'bg_color': '#0d1117',
    'text_color': '#c9d1d9',
    'muted_color': '#8b949e',
}

# Display settings
FIGURE_WIDTH = 1100
FIGURE_HEIGHT = 280

## Load Data

In [12]:
# Import utility functions
sys.path.insert(0, os.path.join(project_root, 'src'))
from utils.time_utils import calculate_time_difference, normalize_time
from utils.geo_utils import haversine

def load_raw_data(year):
    """Load raw train location data from database."""
    conn = sqlite3.connect(DB_PATH)
    query = f"""
    SELECT trip_id, stop_id, vehicle_lat, vehicle_lon, timestamp
    FROM train_locations
    WHERE timestamp LIKE '{year}%'
    """
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    if not df.empty:
        df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
        df = df.dropna(subset=['timestamp'])
    
    print(f"Loaded {len(df):,} raw records for {year}")
    return df

def load_gtfs_data():
    """Load GTFS static data."""
    stops_df = pd.read_csv(os.path.join(GTFS_PATH, 'stops.txt'))
    stops_df = stops_df[stops_df['stop_id'].str.isnumeric()]
    stop_times_df = pd.read_csv(os.path.join(GTFS_PATH, 'stop_times.txt'))
    print(f"Loaded {len(stops_df)} stops, {len(stop_times_df):,} stop times")
    return stops_df, stop_times_df

# Load data
raw_df = load_raw_data(YEAR)
stops_df, stop_times_df = load_gtfs_data()

Base directory: D:\caltrain-tracker
Dotenv path: D:\caltrain-tracker\.env
Dotenv file exists: True
Loaded 1,526,864 raw records for 2025
Loaded 64 stops, 3,694 stop times


## Process Arrivals and Calculate Delays

In [13]:
def process_arrivals(raw_df, stops_df, stop_times_df):
    """Process raw data to calculate arrival delays."""
    if raw_df.empty:
        return pd.DataFrame()
    
    # Ensure consistent types
    raw_df = raw_df.copy()
    raw_df['stop_id'] = pd.to_numeric(raw_df['stop_id'].astype(str), errors='coerce').astype('Int64')
    raw_df['trip_id'] = pd.to_numeric(raw_df['trip_id'].astype(str), errors='coerce').astype('Int64')
    
    stops_df = stops_df.copy()
    stops_df['stop_id'] = pd.to_numeric(stops_df['stop_id'].astype(str), errors='coerce').astype('Int64')
    
    stop_times_df = stop_times_df.copy()
    stop_times_df['stop_id'] = pd.to_numeric(stop_times_df['stop_id'].astype(str), errors='coerce').astype('Int64')
    stop_times_df['trip_id'] = pd.to_numeric(stop_times_df['trip_id'].astype(str), errors='coerce').astype('Int64')
    
    raw_df = raw_df.dropna(subset=['trip_id', 'stop_id'])
    print(f"Processing {len(raw_df):,} records...")
    
    # Merge with schedule and stops
    df = pd.merge(raw_df, stop_times_df[['trip_id', 'stop_id', 'arrival_time']], 
                  on=['trip_id', 'stop_id'], how='inner')
    df = pd.merge(df, stops_df[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']], 
                  on='stop_id', how='inner')
    
    print(f"After merge: {len(df):,} records")
    
    # Calculate distance to stop
    df['distance'] = df.apply(lambda row: haversine(
        row['vehicle_lat'], row['vehicle_lon'], row['stop_lat'], row['stop_lon']
    ), axis=1)
    
    df['date'] = df['timestamp'].dt.date
    df['arrival_time'] = df['arrival_time'].apply(normalize_time)
    
    # Find closest approach (arrival detection)
    min_distances = df.groupby(['trip_id', 'stop_id', 'date'])['distance'].min().reset_index()
    merged = pd.merge(df, min_distances, on=['trip_id', 'stop_id', 'date', 'distance'])
    arrivals = merged.groupby(['trip_id', 'stop_id', 'date']).first().reset_index()
    arrivals = arrivals[['trip_id', 'stop_id', 'date', 'timestamp', 'arrival_time', 'stop_name']]
    arrivals.rename(columns={'timestamp': 'actual_time'}, inplace=True)
    
    # Calculate delay
    arrivals['delay_min'] = arrivals.apply(
        lambda row: calculate_time_difference(row['arrival_time'], row['actual_time']), axis=1
    )
    
    # Clean unrealistic values
    arrivals.loc[arrivals.delay_min > 500, 'delay_min'] = 0
    arrivals.loc[arrivals.delay_min < -100, 'delay_min'] = 0
    
    # Categorize
    arrivals['status'] = 'On Time'
    arrivals.loc[(arrivals.delay_min > 4) & (arrivals.delay_min <= 15), 'status'] = 'Minor'
    arrivals.loc[arrivals.delay_min > 15, 'status'] = 'Major'
    
    print(f"Calculated {len(arrivals):,} arrivals")
    return arrivals

arrivals_df = process_arrivals(raw_df, stops_df, stop_times_df)
arrivals_df['status'].value_counts()

Processing 1,524,149 records...
After merge: 1,523,542 records
Calculated 243,075 arrivals


status
On Time    224212
Minor       16035
Major        2828
Name: count, dtype: int64

## Calculate Daily Statistics

In [14]:
def calculate_daily_stats(arrivals_df):
    """Calculate on-time percentage for each day."""
    if arrivals_df.empty:
        return pd.DataFrame()
    
    daily = arrivals_df.groupby('date')['status'].apply(
        lambda x: (x == 'On Time').mean() * 100
    ).reset_index()
    daily.columns = ['date', 'on_time_pct']
    
    counts = arrivals_df.groupby('date').size().reset_index(name='count')
    daily = daily.merge(counts, on='date')
    daily['date'] = pd.to_datetime(daily['date'])
    
    print(f"Stats for {len(daily)} days | Avg: {daily['on_time_pct'].mean():.1f}%")
    return daily

daily_stats = calculate_daily_stats(arrivals_df)
daily_stats.head(10)

Stats for 141 days | Avg: 92.1%


,date,on_time_pct,count
0,2025-08-10,86.889693,1106
1,2025-08-11,94.401244,1929
2,2025-08-12,92.756644,1919
3,2025-08-13,92.291774,1933
4,2025-08-14,91.455205,1931
5,2025-08-15,91.791430,1937
6,2025-08-16,89.413448,1398
7,2025-08-17,93.366619,1402
8,2025-08-18,75.487013,1848
9,2025-08-19,94.573643,1935


## Create Calendar Heatmap

In [15]:
def prepare_calendar(daily_stats, year):
    """Prepare calendar grid data."""
    start = pd.Timestamp(f'{year}-01-01')
    end = pd.Timestamp(f'{year}-12-31')
    all_dates = pd.date_range(start=start, end=end, freq='D')
    
    cal = pd.DataFrame({'date': all_dates})
    
    if not daily_stats.empty:
        daily_stats = daily_stats.copy()
        daily_stats['date'] = pd.to_datetime(daily_stats['date'])
        cal = cal.merge(daily_stats, on='date', how='left')
    else:
        cal['on_time_pct'] = np.nan
        cal['count'] = 0
    
    cal['dow'] = cal['date'].dt.dayofweek
    first_day = cal['date'].iloc[0]
    cal['week'] = ((cal['date'] - first_day).dt.days + first_day.dayofweek) // 7
    
    return cal

def create_heatmap(cal_data, year):
    """Create the calendar heatmap figure."""
    num_weeks = cal_data['week'].max() + 1
    values = np.full((7, num_weeks), np.nan)
    hovers = [['' for _ in range(num_weeks)] for _ in range(7)]
    
    for _, row in cal_data.iterrows():
        w, d = int(row['week']), int(row['dow'])
        if pd.notna(row['on_time_pct']):
            values[d, w] = row['on_time_pct']
            hovers[d][w] = (
                f"<b>{row['date'].strftime('%A, %b %d, %Y')}</b><br>"
                f"On-Time: <b>{row['on_time_pct']:.1f}%</b><br>"
                f"Arrivals: {int(row['count']) if pd.notna(row['count']) else 0}"
            )
        else:
            values[d, w] = -1
            hovers[d][w] = f"<b>{row['date'].strftime('%A, %b %d, %Y')}</b><br>No data"
    
    day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    
    # Month labels
    month_pos, month_labels = [], []
    current_month = None
    for _, row in cal_data.iterrows():
        m = row['date'].strftime('%b')
        if m != current_month:
            month_pos.append(row['week'])
            month_labels.append(m)
            current_month = m
    
    fig = go.Figure(data=go.Heatmap(
        z=values,
        x=list(range(num_weeks)),
        y=day_labels,
        text=hovers,
        hoverinfo='text',
        colorscale=COLORSCALE,
        zmin=-1,
        zmax=100,
        colorbar=dict(
            title=dict(text='On-Time %', font=dict(color=THEME['text_color'])),
            thickness=15,
            len=0.6,
            ticksuffix='%',
            tickvals=[0, 25, 50, 75, 100],
            tickfont=dict(color=THEME['text_color']),
        ),
        xgap=3,
        ygap=3,
    ))
    
    # Calculate summary
    valid = cal_data[cal_data['on_time_pct'].notna()]
    if not valid.empty:
        avg = valid['on_time_pct'].mean()
        total = valid['count'].sum()
        days = len(valid)
        subtitle = f"Average: {avg:.1f}% on-time | {days} days | {int(total):,} arrivals"
    else:
        subtitle = "No data available"
    
    fig.update_layout(
        title=dict(
            text=f'🚆 Caltrain On-Time Performance {year}<br><span style="font-size:14px;color:{THEME["muted_color"]}">{subtitle}</span>',
            font=dict(size=24, color=THEME['text_color']),
            x=0.5,
            xanchor='center'
        ),
        xaxis=dict(
            tickmode='array',
            tickvals=month_pos,
            ticktext=month_labels,
            tickfont=dict(size=12, color=THEME['muted_color']),
            side='top',
            showgrid=False,
        ),
        yaxis=dict(
            tickfont=dict(size=11, color=THEME['muted_color']),
            showgrid=False,
            autorange='reversed',
        ),
        plot_bgcolor=THEME['bg_color'],
        paper_bgcolor=THEME['bg_color'],
        height=FIGURE_HEIGHT,
        width=FIGURE_WIDTH,
        margin=dict(l=50, r=120, t=100, b=30),
    )
    
    return fig

cal_data = prepare_calendar(daily_stats, YEAR)
fig = create_heatmap(cal_data, YEAR)
fig.show()

## Summary Statistics

In [16]:
def show_summary(daily_stats):
    """Display summary statistics."""
    if daily_stats.empty:
        print("No data")
        return
    
    valid = daily_stats[daily_stats['on_time_pct'].notna()].copy()
    
    print("=" * 60)
    print(f"🚆 CALTRAIN ON-TIME PERFORMANCE - {YEAR}")
    print("=" * 60)
    print(f"\n📅 Days with data: {len(valid)}")
    print(f"🚉 Total arrivals: {valid['count'].sum():,}")
    print(f"\n📊 Overall on-time: {valid['on_time_pct'].mean():.1f}%")
    
    best_idx = valid['on_time_pct'].idxmax()
    worst_idx = valid['on_time_pct'].idxmin()
    
    print(f"\n✅ Best: {valid.loc[best_idx, 'date'].strftime('%b %d')} ({valid['on_time_pct'].max():.1f}%)")
    print(f"❌ Worst: {valid.loc[worst_idx, 'date'].strftime('%b %d')} ({valid['on_time_pct'].min():.1f}%)")
    
    # By day of week
    valid['dow'] = valid['date'].dt.day_name()
    dow_avg = valid.groupby('dow')['on_time_pct'].mean()
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    dow_avg = dow_avg.reindex([d for d in day_order if d in dow_avg.index])
    
    print("\n📆 By Day of Week:")
    print("-" * 40)
    for day, pct in dow_avg.items():
        bar = "█" * int(pct // 5) + "░" * (20 - int(pct // 5))
        emoji = "🟢" if pct >= 80 else "🟡" if pct >= 60 else "🔴"
        print(f"{emoji} {day:10} {pct:5.1f}% |{bar}|")

show_summary(daily_stats)

🚆 CALTRAIN ON-TIME PERFORMANCE - 2025

📅 Days with data: 141
🚉 Total arrivals: 243,075

📊 Overall on-time: 92.1%

✅ Best: Nov 27 (99.0%)
❌ Worst: Oct 04 (63.7%)

📆 By Day of Week:
----------------------------------------
🟢 Monday      92.7% |██████████████████░░|
🟢 Tuesday     92.7% |██████████████████░░|
🟢 Wednesday   93.3% |██████████████████░░|
🟢 Thursday    92.5% |██████████████████░░|
🟢 Friday      91.7% |██████████████████░░|
🟢 Saturday    90.7% |██████████████████░░|
🟢 Sunday      90.9% |██████████████████░░|


## Export to HTML

In [17]:
# Export
os.makedirs(OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(OUTPUT_DIR, f'on_time_heatmap_{YEAR}.html')
fig.write_html(output_path, include_plotlyjs='cdn')
print(f"✅ Exported to: {output_path}")

✅ Exported to: d:\caltrain-tracker\static\plots\on_time_heatmap_2025.html
